## Run only if first time

In [ ]:
%pip install mglyph

## Create and export any glyph you want here

In [2]:
import mglyph as mg
import sys, os

## First try code with simple_scaled_star

In [ ]:
import math
import random
import numpy as np
import json
import zipfile
import io
from datetime import datetime


def random_color():
    return f"#{random.randint(0, 0xFFFFFF):06x}"


def simple_scaled_star(x: float, canvas: mg.Canvas, color: str) -> None:
    
    canvas.tr.translate(0, mg.lerp(x, 0, 0.05))
    
    radius = mg.lerp(x, 0.01, canvas.ysize / 2)

    vertices = []
    for segment in range(5):
        vertices.append(mg.orbit(canvas.center, segment * 2 * math.pi / 5, radius))
        vertices.append(mg.orbit(canvas.center, (segment + 0.5) * 2 * math.pi / 5,
                         math.cos(2 * math.pi / 5) / math.cos(math.pi / 5) * radius))

    # Draw the star with a random outline color
    canvas.polygon(vertices, width='10p', linecap='round', color=color) 

# Initialize dataset metadata
dataset_info = {
    "name": "Experimental dataset with randomly colored stars",
    "time-of-creation": datetime.now().strftime("%Y-%m-%d"),
    "samples": []
}





with zipfile.ZipFile(data_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for i in range(5):
        color1 = random_color()
        color = random_color()

        while color1 == color:
            color = random_color()

        xvalues = np.random.uniform(0.0, 100.0, 120)

        folder_bytes = mg.export(
            lambda x, canvas: simple_scaled_star(x, canvas, color=color),  
            xvalues=xvalues,
            name=f"Random Colored Star {i+1}", short_name=f'star_{i+1}',
            path=None,
            author="Mohaned Anene BARKALLAH", email="Mohanedanene.barkallah@gmail.com", version="1.0.0"
        )

        
        with zipfile.ZipFile(folder_bytes, 'r') as temp_zip:
            
            metadata = None
            for file_name in temp_zip.namelist():
                if file_name.endswith(".json"):
                    metadata = json.loads(temp_zip.read(file_name).decode())
                    break
            
            if metadata:
                
                for file_name in temp_zip.namelist():
                    if file_name.endswith(".png"):
                        
                        new_file_name = f"Random Colored Star {i+1}-{file_name.split('-')[-1]}"
                        zipf.writestr(new_file_name, temp_zip.read(file_name))

                        
                        img_number = file_name.split('-')[-1].split('.')[0]
                        value = None
                        for img_data in metadata['images']:
                            if img_data[0] == f"{img_number.zfill(3)}.png":
                                value = img_data[1]
                                break
                        
                        if value is not None:
                            dataset_info["samples"].append({
                                "value": value,
                                "file": new_file_name
                            })


with zipfile.ZipFile(data_zip, 'a', zipfile.ZIP_DEFLATED) as zipf:
    zipf.writestr('_dataset-info.json', json.dumps(dataset_info, indent=4))


output_path = "data.zip"
with open(output_path, "wb") as f:
    f.write(data_zip.getvalue())

print(f"Dataset successfully saved to {output_path}")

## Generalized code for any glyph drawing function

In [ ]:
import numpy as np
import json
import zipfile
import io
from datetime import datetime

def create_glyph_dataset(
    glyph_function,
    dataset_name,
    num_variants=5,
    samples_per_variant=120,
    value_range=(0.0, 100.0),
    output_path="data.zip",
    author_info=None
):
      """
    Creates a dataset ZIP file from any glyph-drawing function.
    
    Parameters:
    - glyph_function: Function that takes (x, canvas) and draws a glyph
    - dataset_name: Name for the dataset
    - num_variants: Number of glyph variants to generate
    - samples_per_variant: Number of samples per variant
    - value_range: Tuple of (min, max) for parameter values
    - output_path: Where to save the ZIP file
    - author_info: Dict with 'author', 'email', 'version' (optional)
    """
    #Default author
    author_info = author_info or {
        "author": "Unknown",
        "email": "",
        "version": "1.0.0"
    }

    dataset_info = {
        "name": dataset_name,
        "time-of-creation": datetime.now().isoformat(),
        "samples": []
    }

    data_zip = io.BytesIO()

    with zipfile.ZipFile(data_zip, 'w') as zipf:
        for i in range(num_variants):
            # Generate and export variants
            xvalues = np.random.uniform(*value_range, samples_per_variant)
            folder_bytes = mg.export(
                glyph_function,
                xvalues=xvalues,
                name=f"{dataset_name} {i+1}",
                short_name=f'glyph_{i+1}',
                path=None,
                **author_info
            )

            # Process files 
            with zipfile.ZipFile(folder_bytes) as temp_zip:
                # Find metadata
                json_files = [f for f in temp_zip.namelist() if f.endswith('.json')]
                if json_files:
                    metadata = json.loads(temp_zip.read(json_files[0]).decode())
                    
                    # Process images 
                    png_files = [f for f in temp_zip.namelist() if f.endswith('.png')]
                    for file_name in png_files:
                        img_num = file_name.split('-')[-1].split('.')[0]
                        new_name = f"{dataset_name} {i+1}-{file_name.split('-')[-1]}"
                        zipf.writestr(new_name, temp_zip.read(file_name))
                        # Find matching value
                        target = f"{img_num.zfill(3)}.png"
                        value = next((img[1] for img in metadata['images'] if img[0] == target), None)
                        if value is not None:
                            dataset_info["samples"].append({"value": value, "file": new_name})

    # Finalize ZIP
    with zipfile.ZipFile(data_zip, 'a') as zipf:
        zipf.writestr('_dataset-info.json', json.dumps(dataset_info))

    with open(output_path, "wb") as f:
        f.write(data_zip.getvalue())

    return output_path

In [31]:
def random_color():
    return f"#{random.randint(0, 0xFFFFFF):06x}"

def blue_scaled_square(x: float, canvas: mg.Canvas) -> None:
    top = mg.lerp(x, canvas.ycenter, canvas.ytop)
    bottom = mg.lerp(x, canvas.ycenter, canvas.ybottom)
    left = mg.lerp(x, canvas.xcenter, canvas.xleft)
    right = mg.lerp(x, canvas.xcenter, canvas.xright)
    
    # Create polygon points for square
    points = [
        (left, top),
        (right, top),
        (right, bottom),
        (left, bottom)
    ]
    
    canvas.polygon(points, width='15p', linecap='round', color='blue')

In [ ]:
def yellow_scaled_square(x: float, canvas: mg.Canvas) -> None:
    top = mg.lerp(x, canvas.ycenter, canvas.ytop)
    bottom = mg.lerp(x, canvas.ycenter, canvas.ybottom)
    left = mg.lerp(x, canvas.xcenter, canvas.xleft)
    right = mg.lerp(x, canvas.xcenter, canvas.xright)
    
    # Create polygon points for square
    points = [
        (left, top),
        (right, top),
        (right, bottom),
        (left, bottom)
    ]
    
    canvas.polygon(points, width='15p', linecap='round', color='yellow')

In [41]:
def green_scaled_square(x: float, canvas: mg.Canvas) -> None:
    top = mg.lerp(x, canvas.ycenter, canvas.ytop)
    bottom = mg.lerp(x, canvas.ycenter, canvas.ybottom)
    left = mg.lerp(x, canvas.xcenter, canvas.xleft)
    right = mg.lerp(x, canvas.xcenter, canvas.xright)
    
    # Create polygon points for square
    points = [
        (left, top),
        (right, top),
        (right, bottom),
        (left, bottom)
    ]
    
    canvas.polygon(points, width='15p', linecap='round', color='green')

In [30]:
create_glyph_dataset(
    glyph_function=simple_scaled_square,
    dataset_name="Colored Squares",
    num_variants=3,
    author_info={
        "author": "Mohaned Anene BARKALLAH",
        "email": "Mohanedanene.barkallah@gmail.com",
        "version": "1.0.0"
    }
)

IntProgress(value=0, description='Exporting Colored Squares 1 1.0.0:', max=120, style=ProgressStyle(bar_color=…

Exporting Colored Squares 1 1.0.0 finished!


IntProgress(value=0, description='Exporting Colored Squares 2 1.0.0:', max=120, style=ProgressStyle(bar_color=…

Exporting Colored Squares 2 1.0.0 finished!


IntProgress(value=0, description='Exporting Colored Squares 3 1.0.0:', max=120, style=ProgressStyle(bar_color=…

Exporting Colored Squares 3 1.0.0 finished!


'data.zip'

# Zipper class

In [ ]:
import numpy as np
import json
import zipfile
import io
from datetime import datetime

class GlyphExporter:
    def __init__(self, dataset_name="Glyph Dataset"):
        self.dataset_name = dataset_name
        self.glyph_data = []
        self.metadata = {
            "name": dataset_name,
            "time-of-creation": datetime.now().isoformat(),
            "samples": []
        }
        self.author = {
            "author": "Unknown",
            "email": "",
            "version": "1.0.0"
        }

    def export_glyph(self, draw_func, variant_name, xvalues=None, num_samples=120, value_range=(0.0, 100.0)):
        """Export a glyph variant with optional custom parameters"""
        xvalues = (np.array(xvalues) if isinstance(xvalues, (list, tuple))
                  else np.random.uniform(*value_range, num_samples) if xvalues is None
                  else xvalues)
        
        blob = mg.export(
            draw_func,
            xvalues=xvalues,
            name=variant_name,
            short_name=variant_name.lower().replace(" ", "_"),
            path=None,
            **self.author
        )
        self.glyph_data.append((variant_name, blob, xvalues))
        return blob, xvalues

    def finalize_zip(self, output_path="glyphs.zip"):
        """Package all glyphs into a valid ZIP file"""
        with zipfile.ZipFile(output_path, 'w') as zipf:
            with io.BytesIO() as buffer:
                # First write all glyph data to buffer
                for name, blob, _ in self.glyph_data:
                    with zipfile.ZipFile(blob) as glyph_zip:
                        for file in glyph_zip.namelist():
                            if file.endswith('.png'):
                                new_name = f"{name}-{file.split('-')[-1]}"
                                zipf.writestr(new_name, glyph_zip.read(file))
                            elif file.endswith('.json'):
                                data = json.loads(glyph_zip.read(file).decode())
                                self.metadata['samples'].extend({
                                    "value": img[1],
                                    "file": f"{name}-{img[0]}"
                                } for img in data['images'])
                
                # Add metadata
                zipf.writestr('_dataset-info.json', json.dumps(self.metadata, indent=2))

        print(f"Dataset successfully created at {output_path}")
        return output_path

In [52]:
# Usage Example:
xvalues=[0.1*x for x in range(1001)]
exporter = GlyphExporter("My Glyph Collection")

# Export individual glyphs (returns BytesIO but also stores internally)
exporter.export_glyph(blue_scaled_square, "Blue Square") #It can be simple the number of samples will be chosen randomly with random values
exporter.export_glyph(yellow_scaled_square, "yellow square", num_samples=5, value_range=(10.0, 90.0)) #you can set the number of glyphs you want and the value range
exporter.export_glyph(green_scaled_square, "Green square", num_samples=200, xvalues=xvalues) #you can chose the xvalues that you want, it overrides the num_samples if it is larger

# Final packaging
exporter.finalize_zip("data.zip")

IntProgress(value=0, description='Exporting Blue Square 1.0.0:', max=120, style=ProgressStyle(bar_color='cornf…

Exporting Blue Square 1.0.0 finished!


IntProgress(value=0, description='Exporting yellow square 1.0.0:', max=5, style=ProgressStyle(bar_color='cornf…

Exporting yellow square 1.0.0 finished!


IntProgress(value=0, description='Exporting Green square 1.0.0:', max=1001, style=ProgressStyle(bar_color='cor…

Exporting Green square 1.0.0 finished!
Dataset successfully created at data.zip


'data.zip'